In [ ]:
#| default_exp model.models

In [13]:
#| export 
import torch
import torch.nn as nn
import torch.nn.functional as F
import clip
from omegaconf import DictConfig

def make_model(
    config: DictConfig,
):

    if config.model.name == "clip_vitb_16":
        
        model, preprocess = clip.load(
            "ViT-B/16",
            device=config.device,
            jit=False,
        )

        return model, preprocess

In [21]:
#| export
def encode_image_tokens(model, image):

    """All visual tokens of a CLIP ViT: [B, 1 + grid**2, embed_dim]. Token 0 is CLS (== model.encode_image)."""

    v = model.visual
    x = v.conv1(image.type(model.dtype))                                   # [B, width, grid, grid]
    x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)             # [B, grid**2, width]
    cls = v.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device)
    x = torch.cat([cls, x], dim=1) + v.positional_embedding.to(x.dtype)
    x = v.ln_pre(x)
    x = v.transformer(x.permute(1, 0, 2)).permute(1, 0, 2)                 # NLD -> LND -> NLD
    x = v.ln_post(x)                                                       # all tokens, not just x[:, 0]
    return x @ v.proj


def encode_text_tokens(model, text):

    """All text tokens of CLIP: [B, context_length, embed_dim]. The global feature (== model.encode_text) is the
    EOT token at text.argmax(-1); positions after it are padding."""
    
    x = model.token_embedding(text).type(model.dtype) + model.positional_embedding.type(model.dtype)
    x = model.transformer(x.permute(1, 0, 2)).permute(1, 0, 2)             # NLD -> LND -> NLD
    x = model.ln_final(x).type(model.dtype)
    return x @ model.text_projection                                       # all positions, not just EOT

In [14]:
config = DictConfig({
    "device": "cuda", 
    "model": {"name": "clip_vitb_16"}
})

model, preprocess = make_model(config)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          

In [19]:
image = torch.rand(1, 3, 224, 224).to("cuda")
text_tokens = torch.randint(0, 10000, (1, 77)).to("cuda")

image_features = model.encode_image(image)
text_features = model.encode_text(text_tokens)

image_features.shape, text_features.shape

(torch.Size([1, 512]), torch.Size([1, 512]))

In [22]:
image = torch.rand(2, 3, 224, 224).to("cuda")
text = clip.tokenize(["a white suv with a black sunroof", "red sedan"]).to("cuda")

with torch.no_grad():
    
    image_tokens = encode_image_tokens(model, image)
    text_tokens = encode_text_tokens(model, text)
    eot = text.argmax(dim=-1)
    text_mask = torch.arange(text.shape[1], device=text.device) <= eot[:, None]  # True for real tokens

    # The pooled tokens are exactly what CLIP's own encoders return (fp16 on GPU, hence the tolerance).
    assert torch.allclose(image_tokens[:, 0], model.encode_image(image), atol=1e-2)
    assert torch.allclose(text_tokens[torch.arange(len(text)), eot], model.encode_text(text), atol=1e-2)

image_tokens.shape, text_tokens.shape, text_mask.sum(dim=1)

(torch.Size([2, 197, 512]),
 torch.Size([2, 77, 512]),
 tensor([10,  4], device='cuda:0'))